In [1]:
d_model = 64        # hidden size
history_len = 30
forecast_len = 7


GRN (Gated Residual Network)

Encounters: Inside the VSN.
What it does: Smart non-linear processing.

Action: It's a building block. It takes an input vector and:
Tries to process it with dense layers + ELU (non-linearity).
Uses a Gate (sigmoid) to decide: "Should I use this processed version, or just keep the original input?"

Why: It allows the network to learn complex patterns when needed, or just pass simple signals through untouched.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GRN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, output_dim)
        self.fc2 = nn.Linear(output_dim, output_dim)

        self.gate = nn.Linear(output_dim, output_dim)
        self.layer_norm = nn.LayerNorm(output_dim)

        self.project_residual = (
            nn.Linear(input_dim, output_dim)
            if input_dim != output_dim else None
        )

    def forward(self, x):
        residual = x if self.project_residual is None else self.project_residual(x)

        x = F.elu(self.fc1(x))
        x = self.fc2(x)

        gate = torch.sigmoid(self.gate(x))
        x = gate * x + (1 - gate) * residual

        return self.layer_norm(x)

In [3]:
class QuantileLoss(nn.Module):
    """
    Pinball / Quantile loss supporting multiple quantiles.

    Args:
        quantiles: list of floats in (0, 1).
                   Default [0.1, 0.5, 0.9]
                   q=0.5 is the median and gives the main point forecast.
    """
    def __init__(self, quantiles=None):
        super().__init__()
        if quantiles is None:
            quantiles = [0.1, 0.5, 0.9]
        self.register_buffer(
            "quantiles",
            torch.tensor(quantiles, dtype=torch.float32)
        )

    def forward(self, preds, target):
        """
        preds  : [B, T]   — single-output model (current architecture)
        target : [B, T]

        The pinball loss is computed for *each* quantile against the same
        prediction and then averaged.  This keeps the current output head
        unchanged while still training with the correct asymmetric loss.
        """
        errors = target - preds              # [B, T]
        losses = []
        for q in self.quantiles:
            l = torch.where(
                errors >= 0,
                q * errors,
                (q - 1) * errors
            )
            losses.append(l)
        return torch.stack(losses, dim=-1).mean()


VariableSelectionNetwork (VSN)
Called Third: self.past_vsn(...) and self.future_vsn(...)

Role: The Smart Filter.

Action: It looks at the separate feature embeddings from the FeatureEncoder and asks: "Which of these features are actually useful right now?"
It uses a Weight Generator (a sub-network) to assign an importance score (e.g., Price: 0.8, Temp: 0.1, Noise: 0.1).
It creates a weighted combination of the features.

Result: A single, clean vector for each time step containing only the relevant information.
Sub-class used: GRN (Gated Residual Network)
Used by the VSN to process features and generate weights.
It allows the network to apply non-linear processing (using ELU activation) or skip processing entirely (using a Gate), giving it flexibility for both simple and complex patterns.

In [4]:
class VariableSelectionNetwork(nn.Module):
    def __init__(self, num_features, d_model):
        super().__init__()
        self.num_features = num_features

        self.feature_grns = nn.ModuleList(
            [GRN(d_model, d_model) for _ in range(num_features)]
        )

        self.weight_grn = GRN(d_model, d_model)
        self.weight_layer = nn.Linear(d_model, num_features)

    def forward(self, x):
        """
        x: [B, T, F, D]
        """
        B, T, F, D = x.shape

        # Transform each feature separately
        transformed = []
        for i in range(F):
            feat = x[:, :, i, :]  # [B, T, D]
            transformed.append(self.feature_grns[i](feat))

        transformed = torch.stack(transformed, dim=2)  # [B, T, F, D]

        # Compute weights
        context = transformed.mean(dim=2)  # [B, T, D]
        weight_context = self.weight_grn(context)
        weights = self.weight_layer(weight_context)  # [B, T, F]
        weights = torch.softmax(weights, dim=-1)

        # Weighted sum
        output = (transformed * weights.unsqueeze(-1)).sum(dim=2)

        return output

StaticEncoder
Called First: self.static_encoder(static)

Role: Metadata Embedder.

Action: It takes your unchanging "static" data (like Store ID or Item ID) and transforms it into a feature vector (d_model).
Why: This gives the model "context." The model now knows what it is predicting for, before it even looks at the time-series history.

In [5]:
class StaticEncoder(nn.Module):
    def __init__(self, input_dim, d_model):
        super().__init__()
        self.encoder = nn.Linear(input_dim, d_model)

    def forward(self, x):
        return self.encoder(x)


FeatureEncoder 

What it does: Independent projecting.

Action: It takes your time-series data (e.g., 4 features like Price, Temp, Sales). It has 4 separate tiny distinct neural networks (Linear layers) inside it.
Feature 1 goes through Network 1.
Feature 2 goes through Network 2.
...

Output: [Batch, Time, 4, 64] — Each feature is now a rich vector of size 64, but they are still separate.

In [6]:
class FeatureEncoder(nn.Module):
    def __init__(self, num_features, d_model):
        super().__init__()
        self.encoders = nn.ModuleList([
            nn.Linear(1, d_model) for _ in range(num_features)
        ])

    def forward(self, x):
        """
        x: [B, T, F]
        returns: [B, T, F, d_model]
        """
        outs = []
        for i, enc in enumerate(self.encoders):
            outs.append(enc(x[:, :, i:i+1]))
        return torch.stack(outs, dim=2)


In [7]:
import torch.nn as nn


class TemporalAttention(nn.Module):
    """
    Multi-head self-attention across the time dimension.

    Receives the concatenated past + future VSN output of shape [B, T, D],
    lets every time step attend to every other time step, then applies a
    residual connection and LayerNorm for training stability.
    """

    def __init__(self, d_model: int, num_heads: int = 4):
        super().__init__()
        # batch_first=True so tensors are [B, T, D] throughout
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, x):
        """
        Args:
            x: [B, T, D]  –  output from the VSN stage
        Returns:
            [B, T, D]     –  context-enriched representation
        """
        attn_out, _ = self.attn(x, x, x)       # self-attention (Q = K = V = x)
        return self.layer_norm(x + attn_out)    # residual + norm


MinimalTFT (The Conductor)

Role: This is the main class that orchestrates the entire process.

Action: It accepts the raw data (static, past, future) and passes it stage-by-stage through the other components. It manages the flow from encoding $\to$ variable selection $\to$ temporal attention $\to$ prediction.

In [8]:
class MinimalTFT(nn.Module):
    def __init__(self,
                 static_dim,
                 past_dim,
                 future_dim,
                 d_model=64,
                 num_heads=4):
        super().__init__()

        # Encoders
        self.static_encoder = StaticEncoder(static_dim, d_model)
        self.past_encoder = FeatureEncoder(past_dim, d_model)
        self.future_encoder = FeatureEncoder(future_dim, d_model)

        # Variable selection
        self.past_vsn = VariableSelectionNetwork(past_dim, d_model)
        self.future_vsn = VariableSelectionNetwork(future_dim, d_model)

        # Attention
        self.temporal_attn = TemporalAttention(d_model, num_heads)

        # Output head
        self.output_layer = nn.Linear(d_model, 1)

        

    def forward(self, static, past, future):
        """
        static: [B, S]
        past:   [B, T_hist, O]
        future: [B, T_hist + T_fut, K]
        """

        B, T_hist, O = past.shape
        T_total = future.shape[1]

        # Encode
        static_emb = self.static_encoder(static)

        past_emb = self.past_encoder(past)
        future_emb = self.future_encoder(future)


        past_sel = self.past_vsn(past_emb)
        future_sel = self.future_vsn(future_emb)

        # Combine timeline
        temporal_input = torch.cat([past_sel, future_sel[:, T_hist:]], dim=1)

        # Attention
        attn_out = self.temporal_attn(temporal_input)

        # Forecast
        forecast = self.output_layer(attn_out[:, -forecast_len:])
        return forecast.squeeze(-1)


In [9]:
B = 4
static = torch.randn(B, 5)
past = torch.randn(B, 30, 1)
future = torch.randn(B, 37, 4)

model = MinimalTFT(5, 1, 4)
out = model(static, past, future)

print(out.shape)  # [B, 7]


torch.Size([4, 7])


Raw Inputs
[B, T, F]
    ->
FeatureEncoder
[B, T, F, D]
    ->
VariableSelectionNetwork
[B, T, D]
    ->
TemporalAttention
[B, T, D]
    ->
Output Head
[B, T_fut]


### Synthetic data generator to check basline


In [10]:
import torch
import numpy as np

def generate_batch(batch_size=32, history=30, forecast=7):
    t_total = history + forecast
    
    base = torch.randn(batch_size, 1) * 5
    
    time = torch.arange(t_total).float()
    season = torch.sin(2 * np.pi * time / 7)

    sales = base + season + 0.1 * torch.randn(batch_size, t_total)
    
    past = sales[:, :history].unsqueeze(-1)
    future_target = sales[:, history:]
    
    # known future feature: day-of-week sine signal
    known = season.unsqueeze(0).repeat(batch_size, 1)
    known = known.unsqueeze(-1)
    
    static = torch.randn(batch_size, 5)
    
    return static, past, known, future_target

In [11]:
model = MinimalTFT(5, 1, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = QuantileLoss()

for step in range(500):
    static, past, known, target = generate_batch()

    optimizer.zero_grad()
    output = model(static, past, known)
    loss = criterion(output, target)

    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(step, loss.item())

0 2.0436997413635254
50 0.5296768546104431
100 0.221016064286232
150 0.09053032100200653
200 0.16452482342720032
250 0.10748807340860367
300 0.05462333559989929
350 0.09442497789859772
400 0.07882104814052582
450 0.061464615166187286


### Compare current TFT with LSTM(knows only past no known feature data)

In [12]:
import torch
import torch.nn as nn

class LSTMForecast(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, forecast_len=7):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, forecast_len)

    def forward(self, past):
        """
        past: [B, T_hist, 1]
        """
        _, (h, _) = self.lstm(past)
        h = h.squeeze(0)          # [B, hidden_dim]
        return self.fc(h)         # [B, forecast_len]

In [13]:
tft_model = MinimalTFT(5, 1, 1)
tft_optimizer = torch.optim.Adam(tft_model.parameters(), lr=1e-3)
criterion = QuantileLoss()

for step in range(500):
    static, past, known, target = generate_batch()

    tft_optimizer.zero_grad()
    output = tft_model(static, past, known)
    loss = criterion(output, target)

    loss.backward()
    tft_optimizer.step()

    if step % 100 == 0:
        print("TFT", step, loss.item())

TFT 0 2.1267313957214355
TFT 100 0.234197735786438
TFT 200 0.13747462630271912
TFT 300 0.06337220966815948
TFT 400 0.07192479074001312


In [14]:
lstm_model = LSTMForecast()
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)

for step in range(500):
    _, past, _, target = generate_batch()

    lstm_optimizer.zero_grad()
    output = lstm_model(past)
    loss = criterion(output, target)

    loss.backward()
    lstm_optimizer.step()

    if step % 100 == 0:
        print("LSTM", step, loss.item())

LSTM 0 2.072514295578003
LSTM 100 0.5168251395225525
LSTM 200 0.2790343761444092
LSTM 300 0.11800911277532578
LSTM 400 0.053669124841690063


TFT converges early, but at 400 steps they almost hav similar loss because the synthetic data is simple even LSTM can learn periodic patterns easily. So attention dont dominate here.


Upgrade the Synthetic Data

Modify generator to include:

1️⃣ Regime change

Seasonality shifts mid-series.

2️⃣ Promotion spikes (known future feature)

Add binary spike days.

3️⃣ Static-based variation

Make different static inputs change amplitude.

This is where:

Variable selection

Static context

Known future features

start to matter.

Why This Matters

Right now:

LSTM can memorize simple periodicity.

But if:

Regime changes

External known features exist

Different entities behave differently

Then:

TFT should become clearly superior.

In [15]:
import torch
import numpy as np

def generate_batch_v2(batch_size=32, history=30, forecast=7):
    T_total = history + forecast
    
    # ---------- STATIC ----------
    static = torch.randn(batch_size, 5)
    amplitude = 5 + static[:, 0:1] * 2  # [B,1]
    
    # ---------- TIME ----------
    t = torch.arange(T_total).float()
    
    weekly = torch.sin(2 * np.pi * t / 7)          # [T]
    monthly = torch.sin(2 * np.pi * t / 30)        # [T]
    
    regime = torch.where(
        t > T_total // 2,
        torch.tensor(1.5),
        torch.tensor(1.0)
    )  # [T]
    
    # ---------- PROMO ----------
    promo = torch.zeros(batch_size, T_total)
    for b in range(batch_size):
        promo_days = torch.randint(0, T_total, (3,))
        promo[b, promo_days] = 1.0
    
    # ---------- SALES ----------
    # Broadcasting works automatically
    base = amplitude * weekly  # [B, T]
    base += 0.5 * monthly      # monthly broadcasts
    
    sales = base * regime      # regime broadcasts
    sales += 8 * promo
    sales += 0.5 * torch.randn(batch_size, T_total)
    
    # ---------- SPLIT ----------
    past = sales[:, :history].unsqueeze(-1)
    target = sales[:, history:]
    
    known = torch.stack([
        weekly.unsqueeze(0).repeat(batch_size, 1),
        monthly.unsqueeze(0).repeat(batch_size, 1),
        promo
    ], dim=-1)
    
    return static, past, known, target

In [16]:
tft_model = MinimalTFT(5, 1, 3)
tft_optimizer = torch.optim.Adam(tft_model.parameters(), lr=1e-3)
criterion = QuantileLoss()

for step in range(500):
    static, past, known, target = generate_batch_v2()

    tft_optimizer.zero_grad()
    output = tft_model(static, past, known)
    loss = criterion(output, target)

    loss.backward()
    tft_optimizer.step()

    if step % 100 == 0:
        print("TFT", step, loss.item())

TFT 0 2.667027711868286
TFT 100 0.4463996887207031
TFT 200 0.3872040808200836
TFT 300 0.2474268227815628
TFT 400 0.3257271945476532


In [17]:
lstm_model = LSTMForecast()
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)

for step in range(500):
    _, past, _, target = generate_batch_v2()

    lstm_optimizer.zero_grad()
    output = lstm_model(past)
    loss = criterion(output, target)

    loss.backward()
    lstm_optimizer.step()

    if step % 100 == 0:
        print("LSTM", step, loss.item())

LSTM 0 2.5229685306549072
LSTM 100 0.9771876335144043
LSTM 200 0.7465773224830627
LSTM 300 0.6375288367271423
LSTM 400 0.3974243402481079


Why TFT wins here

Your new synthetic includes:

✅ Static-dependent amplitude

TFT sees static → scales properly
LSTM doesn’t use static at all.

✅ Promotion spikes (known future feature)

TFT sees promo directly in future window.
LSTM must guess spikes from past.

That’s almost impossible.

✅ Regime change

Attention helps detect:

time shift

pattern boundary

LSTM struggles with longer shifts.

✅ Multi-frequency seasonality

TFT can:

weigh weekly vs monthly differently

LSTM has to encode everything in hidden state.

In [18]:
class MinimalTFT(nn.Module):
    def __init__(self,
                 num_items,
                 num_stores,
                 num_cats,
                 num_states,
                 past_dim,
                 future_dim,
                 forecast_len=7,
                 d_model=64,
                 num_heads=4):
        super().__init__()

        self.forecast_len = forecast_len
        self.d_model = d_model

        # -------- Static Embeddings --------
        self.item_emb = nn.Embedding(num_items, d_model)
        self.store_emb = nn.Embedding(num_stores, d_model)
        self.cat_emb = nn.Embedding(num_cats, d_model)
        self.state_emb = nn.Embedding(num_states, d_model)

        self.static_proj = nn.Linear(d_model * 4, d_model)

        # -------- Encoders --------
        self.past_encoder = FeatureEncoder(past_dim, d_model)
        self.future_encoder = FeatureEncoder(future_dim, d_model)

        # -------- Variable Selection --------
        self.past_vsn = VariableSelectionNetwork(past_dim, d_model)
        self.future_vsn = VariableSelectionNetwork(future_dim, d_model)

        # -------- Temporal Attention --------
        self.temporal_attn1 = TemporalAttention(d_model, num_heads)
        self.dropout = nn.Dropout(p=0.1)
        self.temporal_attn2 = TemporalAttention(d_model, num_heads)

        # -------- Dropout --------
        self.dropout = nn.Dropout(0.1)
        
        # -------- Output Head --------
        self.output_layer = nn.Linear(d_model, 1)

        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, static, past, future):
        """
        static: [B, 4]  (Long)
        past:   [B, T_hist, past_dim]
        future: [B, T_hist + T_fut, future_dim]
        """

        B, T_hist, _ = past.shape
        T_total = future.shape[1]

        # -------- Static Embedding --------
        item_vec = self.item_emb(static[:, 0])
        store_vec = self.store_emb(static[:, 1])
        cat_vec = self.cat_emb(static[:, 2])
        state_vec = self.state_emb(static[:, 3])

        static_concat = torch.cat(
            [item_vec, store_vec, cat_vec, state_vec],
            dim=-1
        )

        static_emb = self.static_proj(static_concat)  # [B, d_model]

        # -------- Encode Temporal Features --------
        past_emb = self.past_encoder(past)
        future_emb = self.future_encoder(future)

        past_sel = self.past_vsn(past_emb)
        future_sel = self.future_vsn(future_emb)



        # Combine past + future forecast window
        temporal_input = torch.cat(
            [past_sel, future_sel[:, T_hist:]],
            dim=1
        )

        # -------- Inject Static Context --------
        static_expanded = static_emb.unsqueeze(1).expand(
            -1,
            temporal_input.size(1),
            -1
        )

        temporal_input = temporal_input + static_expanded

        temporal_input = self.layernorm(temporal_input)

        # -------- Attention --------
        attn_out = self.temporal_attn1(temporal_input)
        attn_out = self.dropout(attn_out)
        attn_out = self.temporal_attn2(attn_out)
        
        # -------- Forecast --------
        forecast = self.output_layer(attn_out[:, -self.forecast_len:])
        return forecast.squeeze(-1)



In [19]:
import pandas as pd
import numpy as np

sales = pd.read_csv("sales_train_validation.csv")
calendar = pd.read_csv("calendar.csv")
calendar['date'] = pd.to_datetime(calendar['date'])
calendar['day_of_week'] = calendar['date'].dt.weekday
calendar['week_of_year'] = calendar['date'].dt.isocalendar().week.astype(int)
calendar['month'] = calendar['date'].dt.month
calendar['is_weekend'] = calendar['day_of_week'].isin([5,6]).astype(int)
calendar['snap'] = (
    calendar['snap_CA'] |
    calendar['snap_TX'] |
    calendar['snap_WI']
).astype(int)


In [20]:
num_items = sales["item_id"].nunique()
num_stores = sales["store_id"].nunique()
num_cats = sales["cat_id"].nunique()
num_states = sales["state_id"].nunique()
num_items,num_stores,num_cats,num_states

(3049, 10, 3, 3)

In [21]:
# Select first 200 series only
sales = sales.iloc[:200]

We select:
- A subset of series (e.g., first 200).
- The last 400 days of sales data.

This keeps the dataset computationally manageable while preserving recent temporal patterns.

In [22]:
sales_values = sales.iloc[:, -400:].values   # shape: [200, 400]

In [23]:
calendar_subset = calendar.iloc[-400:]

In [24]:
known_cols = [
    "day_of_week",
    "week_of_year",
    "month",
    "is_weekend",
    "snap"
]
known_features = calendar_subset[known_cols].values

From the calendar dataset, we extract known future features such as:

- `wday` (weekday index)
- `month`
- `snap_CA`, `snap_TX`, `snap_WI`

These features are known ahead of time and will be provided to the model for both the historical and forecast window.

In [25]:
means = sales_values.mean(axis=1, keepdims=True)
stds = sales_values.std(axis=1, keepdims=True) + 1e-6

sales_norm = (sales_values - means) / stds

Retail demand varies significantly across products.

We normalize each time series independently:

\[
x_{norm} = \frac{x - \mu}{\sigma}
\]

This stabilizes training and prevents high-volume items from dominating gradients.

In [26]:
for col in ["item_id", "store_id", "cat_id", "state_id"]:
    sales[col] = sales[col].astype("category").cat.codes

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
static_features = sales[["item_id", "store_id", "cat_id", "state_id"]].values
static_tensor = torch.tensor(static_features).long().to(device)

In [28]:
import pandas as pd

def compute_past_features(sales_norm):
    """
    Precomputes [N_series, T, 5] feature array:
      dim 0: raw sales (normalized)
      dim 1: 7-day rolling mean
      dim 2: 7-day rolling std
      dim 3: lag-1
      dim 4: lag-7
    NaN boundaries are zero-padded.
    """
    N, T = sales_norm.shape
    features = np.zeros((N, T, 5))

    for i in range(N):
        s = pd.Series(sales_norm[i])

        features[i, :, 0] = s.values                                    # raw
        features[i, :, 1] = s.rolling(7, min_periods=1).mean().values   # rolling mean
        features[i, :, 2] = s.rolling(7, min_periods=1).std().fillna(0).values  # rolling std
        features[i, 1:,  3] = s.values[:-1]                             # lag-1
        features[i, 7:,  4] = s.values[:-7]                             # lag-7
        # positions 0 and 0:7 stay 0-padded (already zeros)

    return features

past_features = compute_past_features(sales_norm)
print(f"past_features shape: {past_features.shape}")   # (200, 400, 5)


past_features shape: (200, 400, 5)


In [29]:
history = 30
forecast = 7

def create_samples(past_features, known_features, sales_norm, history=30, forecast=7):
    """
    Slide a window across all series and collect (series_id, past, known, target) tuples.
    past now has shape [history, 5] — raw + rolling mean/std + lag1/lag7.
    """
    T_total = sales_norm.shape[1]
    samples = []

    for i in range(past_features.shape[0]):
        for t in range(history, T_total - forecast):
            past   = past_features[i, t - history : t, :]   # [30, 5]
            target = sales_norm[i, t : t + forecast]         # [7]
            known  = known_features[t - history : t + forecast]  # [37, 5]
            samples.append((i, past, known, target))

    return samples

samples = create_samples(past_features, known_features, sales_norm)
print(f"Total samples: {len(samples)}")


Total samples: 72600


In [30]:
def split_samples(samples, train_ratio=0.8):
    """
    Split samples into train/val by index (temporal order is preserved).
    """
    split_idx = int(len(samples) * train_ratio)
    return samples[:split_idx], samples[split_idx:]

train_samples, val_samples = split_samples(samples)


In [31]:
def build_tensors(samples):
    series_ids    = torch.tensor([s[0] for s in samples])
    past_tensor   = torch.tensor(np.array([s[1] for s in samples])).float()  # [N, 30, 5]
    known_tensor  = torch.tensor(np.array([s[2] for s in samples])).float()  # [N, 37, 5]
    target_tensor = torch.tensor(np.array([s[3] for s in samples])).float()  # [N, 7]
    return series_ids, past_tensor, known_tensor, target_tensor

train_ids, train_past, train_known, train_target = build_tensors(train_samples)
val_ids,   val_past,   val_known,   val_target   = build_tensors(val_samples)

print("train_past shape:", train_past.shape)    # [N, 30, 5]
print("train_known shape:", train_known.shape)  # [N, 37, 5]
print("train_target shape:", train_target.shape) # [N, 7]


train_past shape: torch.Size([58080, 30, 5])
train_known shape: torch.Size([58080, 37, 5])
train_target shape: torch.Size([58080, 7])


We convert the sliding window samples into PyTorch tensors:

- `past_tensor` → [N, 30, 1]
- `known_tensor` → [N, 37, K]
- `target_tensor` → [N, 7]
- `series_ids` → used to fetch static features

This prepares the dataset for model training.

In [32]:
train_ids = train_ids.to(device)
train_past = train_past.to(device)
train_known = train_known.to(device)
train_target = train_target.to(device)

val_ids = val_ids.to(device)
val_past = val_past.to(device)
val_known = val_known.to(device)
val_target = val_target.to(device)

In [33]:
import numpy as np

def compute_rmsse_denominator(train_raw):
    """
    Computes the denominator (scale) of the RMSSE calculation for each series.
    Follows M5 rules: calculated on the active training period (after the first non-zero sale).
    """
    num_series, T = train_raw.shape
    denom = np.zeros(num_series)
    
    for i in range(num_series):
        series = train_raw[i]
        non_zero_idx = np.where(series > 0)[0]
        
        if len(non_zero_idx) == 0:
            denom[i] = 1.0  
        else:
            first_nz = non_zero_idx[0]
            active_series = series[first_nz:]
            
            if len(active_series) < 2:
                denom[i] = 1.0
            else:
                diffs = np.diff(active_series)
                scale = np.mean(diffs ** 2)
                denom[i] = scale if scale > 0 else 1.0
            
    return denom

def compute_rmsse(y_true_raw, y_pred_raw, denom):
    """
    Computes the RMSSE for each series over the horizon.
    """
    mse = np.mean((y_true_raw - y_pred_raw) ** 2, axis=1)
    return np.sqrt(mse / denom)

def compute_wrmsse(rmsse, weights):
    """
    Computes the Weighted RMSSE across all bottom-level series.
    """
    return float(np.sum(rmsse * weights))

# Pre-calculate the scale denominator using your raw `sales_values` array [200, 400]
rmsse_denoms = compute_rmsse_denominator(sales_values)

# Uniform weights for bottom-level only (since there are 200 series currently)
num_series = sales_values.shape[0]
bottom_weights = np.ones(num_series) / num_series


In [34]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------
# MODEL
# -----------------------
model = MinimalTFT(
    num_items=num_items,
    num_stores=num_stores,
    num_cats=num_cats,
    num_states=num_states,
    past_dim=5,       
    future_dim=len(known_cols),
    forecast_len=7,   
    d_model=192,
    num_heads=8,
).to(device)


# -----------------------
# OPTIMIZER + LOSS
# -----------------------
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
criterion = QuantileLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=3,
    factor=0.5
)

batch_size = 64
num_epochs = 30
patience = 6
best_val_loss = float("inf")
early_stop_counter = 0

# -----------------------
# DEBUG: Check target scale
# -----------------------
print("Target min:", train_target.min().item())
print("Target max:", train_target.max().item())

# -----------------------
# TRAINING LOOP
# -----------------------
for epoch in range(num_epochs):

    model.train()
    perm = torch.randperm(len(train_past))

    total_train_loss = 0
    num_batches = 0

    for i in range(0, len(train_past), batch_size):

        idx = perm[i:i+batch_size]

        static = static_tensor[train_ids[idx]].to(device)
        past = train_past[idx].to(device)
        known = train_known[idx].to(device)
        target = train_target[idx].to(device)

        optimizer.zero_grad()

        output = model(static, past, known)

        # Debug shape check (only first epoch first batch)
        if epoch == 0 and i == 0:
            print("Output shape:", output.shape)
            print("Target shape:", target.shape)

        loss = criterion(output, target)

        loss.backward()

        # Gradient clipping (important for transformers)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_train_loss += loss.item()
        num_batches += 1

    avg_train_loss = total_train_loss / num_batches

    # -----------------------
    # VALIDATION
    # -----------------------
    model.eval()
    with torch.no_grad():

        static_val = static_tensor[val_ids].to(device)
        val_output = model(static_val, val_past.to(device), val_known.to(device))
        val_loss = criterion(val_output, val_target.to(device))

        # Median MAE: model outputs [B, 7] (single point forecast = median proxy)
        median_mae = torch.mean(torch.abs(val_output - val_target.to(device)))

    scheduler.step(val_loss)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {val_loss.item():.4f} | "
        f"Median MAE: {median_mae.item():.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.6f}"
    )

    # -----------------------
    # EARLY STOPPING
    # -----------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stop_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        early_stop_counter += 1

    if early_stop_counter >= patience:
        print("Early stopping triggered.")
        break


# -----------------------
# FINAL EVALUATION ON FULL VAL SET
# -----------------------
print("\n--- Final Evaluation ---")

model.eval()
with torch.no_grad():
    static_val = static_tensor[val_ids].to(device)
    val_output = model(static_val, val_past.to(device), val_known.to(device))

    # Model output is [B, 7] — this IS the point forecast (median proxy)
    median_mae = torch.mean(torch.abs(val_output - val_target.to(device)))

    # Naive baseline: repeat last known sales value across all 7 forecast days
    last_step = val_past[:, -1, 0].to(device)              # [B]  — raw sales column
    naive_pred = last_step.unsqueeze(1).expand(-1, 7)       # [B, 7]
    naive_mae = torch.mean(torch.abs(naive_pred - val_target.to(device)))

print(f"Naive (last-value) MAE : {naive_mae.item():.4f}")
print(f"TFT Median MAE         : {median_mae.item():.4f}")
print(f"Best Val Quantile Loss : {best_val_loss.item():.4f}")


Target min: -1.782309651374817
Target max: 17.861536026000977
Output shape: torch.Size([64, 7])
Target shape: torch.Size([64, 7])
Epoch 00 | Train Loss: 0.3021 | Val Loss: 0.2912 | Median MAE: 0.5824 | LR: 0.000300
Epoch 01 | Train Loss: 0.2934 | Val Loss: 0.2880 | Median MAE: 0.5760 | LR: 0.000300
Epoch 02 | Train Loss: 0.2918 | Val Loss: 0.2882 | Median MAE: 0.5764 | LR: 0.000300
Epoch 03 | Train Loss: 0.2911 | Val Loss: 0.2898 | Median MAE: 0.5796 | LR: 0.000300
Epoch 04 | Train Loss: 0.2903 | Val Loss: 0.2930 | Median MAE: 0.5861 | LR: 0.000300
Epoch 05 | Train Loss: 0.2902 | Val Loss: 0.2868 | Median MAE: 0.5735 | LR: 0.000300
Epoch 06 | Train Loss: 0.2896 | Val Loss: 0.2860 | Median MAE: 0.5720 | LR: 0.000300
Epoch 07 | Train Loss: 0.2897 | Val Loss: 0.2874 | Median MAE: 0.5747 | LR: 0.000300
Epoch 08 | Train Loss: 0.2890 | Val Loss: 0.2854 | Median MAE: 0.5708 | LR: 0.000300
Epoch 09 | Train Loss: 0.2885 | Val Loss: 0.2849 | Median MAE: 0.5697 | LR: 0.000300
Epoch 10 | Train Los

In [35]:
print("\n--- Final Evaluation ---")

model.eval()
with torch.no_grad():
    static_val = static_tensor[val_ids].to(device)
    val_output = model(static_val, val_past.to(device), val_known.to(device))
    val_target_device = val_target.to(device)

    # Model output is [B, 7] — this IS the point forecast (median proxy)
    median_mae = torch.mean(torch.abs(val_output - val_target_device))

    # Naive baseline: repeat last known sales value across all 7 forecast days
    last_step = val_past[:, -1, 0].to(device)              # [B] — raw normalized sales column
    naive_pred = last_step.unsqueeze(1).expand(-1, 7)       # [B, 7]
    naive_mae = torch.mean(torch.abs(naive_pred - val_target_device))

    # --- NEW: WRMSSE CALCULATION ---
    val_preds_np = val_output.cpu().numpy()
    val_trues_np = val_target_device.cpu().numpy()
    val_ids_np = val_ids.cpu().numpy()

    # Fetch specific mean/std/denoms for each dataset ID
    batch_means = means[val_ids_np]
    batch_stds = stds[val_ids_np]
    batch_denoms = rmsse_denoms[val_ids_np]

    # Denormalize to RAW space
    val_preds_raw = (val_preds_np * batch_stds) + batch_means
    val_trues_raw = (val_trues_np * batch_stds) + batch_means

    # Naive prediction denormalization for baseline WRMSSE
    naive_preds_np = naive_pred.cpu().numpy()
    naive_preds_raw = (naive_preds_np * batch_stds) + batch_means

    # Calculate final batch-wide WRMSSE
    tft_rmsse_values = compute_rmsse(val_trues_raw, val_preds_raw, batch_denoms)
    tft_wrmsse = float(np.mean(tft_rmsse_values))

    naive_rmsse_values = compute_rmsse(val_trues_raw, naive_preds_raw, batch_denoms)
    naive_wrmsse = float(np.mean(naive_rmsse_values))

print(f"Naive MAE (Norm)       : {naive_mae.item():.4f}")
print(f"TFT Median MAE (Norm)  : {median_mae.item():.4f}")
print(f"Best Val Quantile Loss : {best_val_loss.item():.4f}")
print("-" * 30)
print(f"Naive WRMSSE (Raw)     : {naive_wrmsse:.4f}")
print(f"TFT WRMSSE (Raw)       : {tft_wrmsse:.4f}")



--- Final Evaluation ---
Naive MAE (Norm)       : 0.8204
TFT Median MAE (Norm)  : 0.5697
Best Val Quantile Loss : 0.2839
------------------------------
Naive WRMSSE (Raw)     : 0.8088
TFT WRMSSE (Raw)       : 0.6505


In [36]:
# -----------------------
# HORIZON-WISE ERROR
# -----------------------
print("\n--- Horizon-wise MAE ---")

model.eval()
with torch.no_grad():

    static_val = static_tensor[val_ids].to(device)
    preds = model(static_val, val_past.to(device), val_known.to(device))

    errors = torch.abs(preds - val_target.to(device))

    for h in range(7):
        horizon_mae = errors[:, h].mean()
        print(f"Day {h+1} MAE : {horizon_mae.item():.4f}")


--- Horizon-wise MAE ---
Day 1 MAE : 0.5698
Day 2 MAE : 0.5701
Day 3 MAE : 0.5699
Day 4 MAE : 0.5695
Day 5 MAE : 0.5692
Day 6 MAE : 0.5695
Day 7 MAE : 0.5697
